# YOLO instance segmentation with dataset-fixer

Dataset conversion, standardized W&B config/tags, portable bundle manifests, ZIP creation,
and W&B upload are handled by dataset-fixer. The ZIP is always created locally and is never
copied to Google Drive. The final cell prints whether W&B accepted the upload, the run URL,
the remote file URL when available, and the exact local path/hash retained in every case.

In [ ]:
import importlib.util
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    !pip install --upgrade --upgrade-strategy only-if-needed "dataset-fixer @ git+https://github.com/mooch443/dataset-fixer.git"

## Configuration

### Common settings

Change these for a new dataset, model scale, or training run.

In [ ]:
from datetime import datetime
from pathlib import Path
import torch

DATASET_SOURCE = (
    "/content/drive/MyDrive/islands/yolo-instance-seg.zip"
    if IN_COLAB else
    "/Users/tristan/Downloads/island-dataset/yolo-instance-seg"
)
NATIVE_TILE_SIZE = 128
UPSCALE_FACTOR = 2
EPOCHS = 100
BASE_MODEL = "yolo26m-seg.pt"
USE_WANDB = True

### Advanced settings

These defaults control execution and naming and usually do not need editing.

In [ ]:
RESULTS_DIRECTORY = Path("/content/training-results" if IN_COLAB else "./training-results").resolve()
WORKERS = 4
MODEL_INPUT_SIZE = NATIVE_TILE_SIZE * UPSCALE_FACTOR
TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M")

base_model_name = Path(BASE_MODEL).stem
MODEL_NAME = f"{base_model_name}-{MODEL_INPUT_SIZE}px-{UPSCALE_FACTOR}x-{TIMESTAMP}"
TRAINING_NAME = MODEL_NAME
DEVICE_OVERRIDE = None  # None selects CUDA, then MPS, then CPU.
EVALUATION_BATCH_SIZE = -1
WANDB_ENTITY = "max-planck-institute-for-animal-behavior"
WANDB_PROJECT = "schools-segmentation"
WANDB_RUN_NAME = TRAINING_NAME
TRAINING_PROJECT = WANDB_PROJECT if USE_WANDB else "ultralytics"

device = DEVICE_OVERRIDE or (
    "cuda" if torch.cuda.is_available() else
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"
)

In [ ]:
if USE_WANDB:
    !wandb login

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

## Preparation

In [ ]:
from dataset_fixer import Dataset
from dataset_fixer.convert import Kind, prepare

# Dataset.open accepts either a folder or ZIP and reuses a verified local extraction.
# YOLO_SEG accepts only existing polygon instance annotations; semantic masks are never
# converted into polygons. prepare() audits those annotations, scales the images, and
# reuses its content-addressed output when the data and geometry have not changed. The
# run-specific arguments populate prepared.config without changing that cache identity.
source_dataset = Dataset.open(DATASET_SOURCE)
prepared = prepare(
    source_dataset,
    Kind.YOLO_SEG,
    name=MODEL_NAME,
    native_tile_size=NATIVE_TILE_SIZE,
    upscale_factor=UPSCALE_FACTOR,
    workers=WORKERS,
    preprocess=False,
    base_model=BASE_MODEL,
    epochs=EPOCHS,
    device=device,
)
bundle_config = prepared.config
prepared

### Configure W&B and Ultralytics

In [ ]:
import os
from ultralytics import settings

settings.update({"wandb": USE_WANDB})
WANDB_RUN_PATH = None
if USE_WANDB:
    os.environ.update({
        "WANDB_ENTITY": WANDB_ENTITY,
        "WANDB_PROJECT": WANDB_PROJECT,
        "WANDB_NAME": WANDB_RUN_NAME,
    })
    print(f"Ultralytics will sync training to {WANDB_ENTITY}/{WANDB_PROJECT} as {WANDB_RUN_NAME}.")
else:
    print("W&B disabled; the completed bundle will remain local.")

## Train

In [ ]:
import wandb
from dataset_fixer.wandb import configure as configure_wandb

RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)
starting_directory = Path.cwd()
%cd {RESULTS_DIRECTORY}
!yolo segment train model="{BASE_MODEL}" data="{prepared.data_yaml}" imgsz={MODEL_INPUT_SIZE} epochs={EPOCHS} workers={WORKERS} device="{device}" project="{TRAINING_PROJECT}" name="{TRAINING_NAME}"
%cd {starting_directory}

TRAINING_DIRECTORY = RESULTS_DIRECTORY / TRAINING_PROJECT / TRAINING_NAME
CHECKPOINT = (TRAINING_DIRECTORY / "weights" / "best.pt").resolve()
if not CHECKPOINT.is_file():
    raise FileNotFoundError(f"Training completed without expected checkpoint: {CHECKPOINT}")

if USE_WANDB:
    try:
        matching_runs = wandb.Api().runs(
            f"{WANDB_ENTITY}/{WANDB_PROJECT}",
            filters={"display_name": WANDB_RUN_NAME},
            order="-created_at",
        )
        training_run = next(iter(matching_runs), None)
        if training_run is None:
            print(f"W&B training run not found yet for name {WANDB_RUN_NAME!r}; the bundle will remain local.")
        else:
            configure_wandb(training_run, bundle_config)
            WANDB_RUN_PATH = f"{training_run.entity}/{training_run.project}/{training_run.id}"
            print(f"Standardized W&B config written to: https://wandb.ai/{WANDB_RUN_PATH}")
    except Exception as exc:
        print(f"Could not configure the W&B training run: {type(exc).__name__}: {exc}")

## Evaluate results

In [ ]:
from dataset_fixer import Model

models = Model.load_many([CHECKPOINT])
model_name = models.names[0]
models = models.configure({
    model_name: {
        "task": "segment", 
        "native_tile_size": NATIVE_TILE_SIZE,
        "upscale_factor": UPSCALE_FACTOR, "device": device,
        "workers": WORKERS, 
        "batch_size": EVALUATION_BATCH_SIZE,
        "inference": "sahi"
    }
})
comparison = models.compare(source_dataset, split="val", save_prediction_plots=True)

In [ ]:
import wandb
from dataset_fixer.bundle import Outcome, create
from dataset_fixer.wandb import configure as configure_wandb
from dataset_fixer.wandb import upload

outcome = Outcome(checkpoint=CHECKPOINT, metrics={"comparison_report": str(comparison.location)})
model_bundle = create(bundle_config, outcome)
upload_run = None
if WANDB_RUN_PATH is not None:
    try:
        upload_run = wandb.Api().run(WANDB_RUN_PATH)
        configure_wandb(upload_run, bundle_config)
    except Exception as exc:
        print(f"Could not reopen W&B run {WANDB_RUN_PATH}: {type(exc).__name__}: {exc}")
if upload_run is not None:
    model_bundle = upload(upload_run, model_bundle, outcome)
elif USE_WANDB:
    print("W&B upload skipped because the intended run could not be reopened.")
print(f"Local bundle: {model_bundle.path}")
print(f"Size: {model_bundle.size:,} bytes")
print(f"SHA-256: {model_bundle.sha256}")
if model_bundle.uploaded:
    print("W&B sync: confirmed; upload completed and run summary updated.")
    print(f"W&B run: https://wandb.ai/{WANDB_RUN_PATH}")
    print(f"W&B file: {model_bundle.path.name}")
    if model_bundle.remote_url:
        print(f"Remote file: {model_bundle.remote_url}")
else:
    print("W&B sync: not completed; use the local bundle above.")
    for warning in model_bundle.warnings:
        print(f"- {warning}")